## 11.6 התאמה ליניארית: מהנוסחה לקוד

עד עכשיו חישבנו סטטיסטיקות על **קבוצה אחת** של מדידות (ממוצע, סטיית תקן, SEM). עכשיו עוברים לשאלה אחרת: יש לנו **כמה נקודות** -- טווח ממוצע לכל זווית -- ורוצים למצוא את הקשר הליניארי ביניהן ובין `sin(2*theta)`, לפי המודל התאורטי `R = (v0**2/g) * sin(2*theta)`.

זו בדיוק משוואת קו ישר: `y = m*x + b`, כאשר `x = sin(2*theta)`, `y = R`, השיפוע `m` אמור לצאת קרוב ל-`v0**2/g`, וה-חיתוך `b` קרוב ל-0.

בקורס הזה **לא** נשתמש ב-`np.polyfit` או `scipy.optimize.curve_fit` כקופסה שחורה -- נכתוב את פונקציית ההתאמה הליניארית בעצמנו, מהנוסחה הסגורה (least squares), כדי להבין בדיוק מה יש בפנים.

```{admonition} 🎥 כאן נכנס סרטון השבוע
:class: seealso

**"התאמה ליניארית: מהנוסחה לקוד"** (כ-6–10 דקות, הקלטת מסך של המחברת עם קול). בסרטון בונים את פונקציית ההתאמה הליניארית מאפס, שורה אחר שורה, ומיישמים אותה על נתוני הטווח מול sin(2*theta).

<!-- TODO (למפיק/ה): להחליף תא זה בהטמעת הסרטון בפועל, למשל:
<iframe width="100%" height="400" src="VIDEO_URL_HERE" title="שבוע 11 — התאמה ליניארית: מהנוסחה לקוד" frameborder="0" allowfullscreen></iframe>
-->
```

### נוסחת הריבועים הפחותים (least squares)

לקו `y = m*x + b` שממזער את סכום ריבועי השאריות, יש נוסחה סגורה:

$$m = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2} \qquad b = \bar{y} - m\bar{x}$$

זה בדיוק אותו רעיון שראינו בתרגול העצמי -- ממוצעים, סטיות מהממוצע -- הפעם על שני משתנים יחד.

In [ ]:
import numpy as np

def linear_fit(x, y):
    """מחזירה (m, b) - שיפוע וחיתוך של התאמה ליניארית y = m*x + b, בשיטת הריבועים הפחותים."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x_bar = x.mean()
    y_bar = y.mean()
    m = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar)**2)
    b = y_bar - m * x_bar
    return m, b

### יישום על נתוני הזריקה

בונים את `x` (`sin(2*theta)`) ו-`y` (הטווח הממוצע) לכל זווית, מתוך `lab_measurements.csv`.

In [ ]:
import pandas as pd

g = 9.8
df = pd.read_csv("lab_measurements.csv")
df_clean = df.dropna()

angles = sorted(df_clean["angle_deg"].unique())
x = np.array([np.sin(2*np.radians(a)) for a in angles])
y = np.array([df_clean[df_clean["angle_deg"] == a]["range_measured"].mean() for a in angles])

m, b = linear_fit(x, y)
print(f"שיפוע m = {m:.3f}  (צפי תאורטי v0^2/g ~ {20.0**2/g:.3f})")
print(f"חיתוך b = {b:.3f}  (צפי תאורטי ~ 0)")

### באג נפוץ: לשכוח למרכז (centering) את הנתונים

טעות שכיחה: לכתוב `m = sum(x*y) / sum(x*x)` -- זו נוסחת התאמה ליניארית **דרך הראשית** (`b=0` בכפייה), לא ההתאמה הכללית. היא לא זורקת שגיאה, ונותנת מספר סביר -- אבל שגוי, כי היא מתעלמת מהחיתוך.

In [ ]:
m_through_origin = np.sum(x*y) / np.sum(x*x)   # מניח (בטעות) b=0
print(f"שיפוע 'דרך הראשית': {m_through_origin:.3f}")
print(f"שיפוע נכון (עם מרכוז): {m:.3f}")
print("שני מספרים שונים - לא כי אחת השיטות 'טועה טעות גסה', אלא כי הן עונות על שאלות שונות (b=0 בכפייה, מול b חופשי).")

### נסו בעצמכם

הפעילו את `linear_fit` על `run_id` (ציר x) מול `v0_measured` (ציר y) בתוך זווית 45 בלבד. אמור לצאת שיפוע קרוב מאוד ל-0 (אין מגמת שינוי אמיתית לאורך הריצות -- v0 היה אמור להיות קבוע).

In [ ]:
# sub45 = df_clean[df_clean["angle_deg"] == 45]
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
sub45 = df_clean[df_clean["angle_deg"] == 45]
m45, b45 = linear_fit(sub45["run_id"], sub45["v0_measured"])
print(f"שיפוע: {m45:.4f}  (קרוב ל-0, כצפוי - v0 לא אמור להשתנות עם run_id)")
```
`````

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "מה ההבדל בין `m = sum((x-xbar)*(y-ybar)) / sum((x-xbar)**2)` לבין `m = sum(x*y) / sum(x*x)`?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "הראשונה מתאימה קו כללי (b חופשי), השנייה מכריחה את הקו לעבור דרך הראשית (b=0)", "correct": True, "feedback": "נכון."},
            {"answer": "אין הבדל, שתי הנוסחאות שקולות תמיד", "correct": False, "feedback": "לא - הן שקולות רק אם xbar=0 או ybar=0."},
            {"answer": "השנייה היא הנוסחה הנכונה תמיד, הראשונה שגויה", "correct": False, "feedback": "לא - שתיהן נכונות, לשתי שאלות שונות."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

כתבו פונקציה `linear_fit_manual_loop(x, y)` שמחשבת את אותו `m, b` בעזרת **לולאת `for`** מפורשת (סכימה איבר-איבר, בלי `np.sum` ובלי חיסור וקטורי), והשוו את התוצאה ל-`linear_fit` המקורית עם `np.allclose`. זה תרגיל טוב להבין מה בדיוק `np.sum`/וקטוריזציה "חוסכות" לנו.

In [ ]:
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
def linear_fit_manual_loop(x, y):
    x = list(x)
    y = list(y)
    n = len(x)
    x_bar = sum(x) / n
    y_bar = sum(y) / n

    num = 0.0
    den = 0.0
    for i in range(n):
        num += (x[i] - x_bar) * (y[i] - y_bar)
        den += (x[i] - x_bar)**2

    m_loop = num / den
    b_loop = y_bar - m_loop * x_bar
    return m_loop, b_loop

m_loop, b_loop = linear_fit_manual_loop(x, y)
print(np.allclose([m_loop, b_loop], [m, b]))
```
`````